# Mall World: Corrected Functional Hypothesis Testing

**Purpose:** Test hypotheses based on the *functional structure* of the World of Spirits, correcting the errors in the original analysis.

**Key Insight:** The World of Spirits is not a reactive projection of the experiencer's emotional state — it is an **inherent functional structure** through which spirits pass. The reality is fundamentally SOLID because it operates by consistent spiritual laws.

---

## Corrected Framework

### Transit = Source of Guidance
- **Passive transit**: Being guided by external source (angelic, instructive)
- **Active transit**: Self-directed by the experiencer's **ruling love** (their spiritual "east")
- Both succeed because they are *different mechanisms of the same system*

### Locations = Functional Stations
| Location | Spiritual Function |
|----------|--------------------|
| Markets/Malls | Appropriation of goods and truths |
| Libraries | Internal reasoning, rational appropriation |
| Schools | Guidance toward broader truth from what's appropriated |
| Hotels | Passive reflection |
| Hospitals | Intrusive correction |
| Pleasure places | Testing/temptation of appropriations |

### The Illusion Pattern
Some locations appear outwardly good but are **empty inside**:
- Food that is tasteless
- Guidance toward hellish environments after appropriation
- Casinos, buffets — appearance of abundance masking spiritual emptiness

---

## New Hypotheses

1. **H1: Passive Transit → Instructional Destinations** — Passive transit leads to schools, places of rest, teaching environments
2. **H2: Transit Success is Universal** — Both passive and active transit succeed because they're functional mechanisms
3. **H3: Commercial Locations = Appropriation** — Malls, markets, libraries function for acquisition of goods/truths
4. **H4: Hospitals = Corrective Intervention** — Hospital visits correlate with negative states being addressed
5. **H5: Pleasure Places = Testing/Temptation** — Casinos, amusement parks, buffets show the illusion pattern (appear good, reveal emptiness)
6. **H6: Baseline Reality is Solid** — Most locations show stable reality because structure is inherent, not projected

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, binomtest
from collections import Counter
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries loaded")

✓ Libraries loaded


In [2]:
# Load data from previous analysis (using exported CSVs for speed)
data_dir = Path('../reports/data')

if data_dir.exists() and (data_dir / 'locations.csv').exists():
    df_dreams = pd.read_csv(data_dir / 'dreams_meta.csv')
    df_locations = pd.read_csv(data_dir / 'locations.csv')
    df_connections = pd.read_csv(data_dir / 'connections.csv')
    df_interactions = pd.read_csv(data_dir / 'interactions.csv')
    df_entities = pd.read_csv(data_dir / 'entities.csv')
    print("✓ Loaded from cached CSVs")
else:
    print("⚠️ Need to run original notebook first to export data")
    
print(f"\nData Summary:")
print(f"  Dreams:       {len(df_dreams):,}")
print(f"  Locations:    {len(df_locations):,}")
print(f"  Connections:  {len(df_connections):,}")
print(f"  Interactions: {len(df_interactions):,}")
print(f"  Entities:     {len(df_entities):,}")

✓ Loaded from cached CSVs

Data Summary:
  Dreams:       1,926
  Locations:    8,707
  Connections:  4,600
  Interactions: 6,075
  Entities:     3,704


In [3]:
# Define functional location categories based on corrected framework

# APPROPRIATION ZONES - places of acquiring goods and truths
APPROPRIATION_LOCATIONS = {
    'mall', 'mall_store', 'mall_food_court',
    'market', 'grocery_store', 'shop',
    'library',  # rational appropriation through reasoning
}

# INSTRUCTION/GUIDANCE - places of teaching and broader truth
INSTRUCTION_LOCATIONS = {
    'school', 'school_classroom', 'school_library', 'school_cafeteria', 'school_gym',
    'university', 'lecture_hall',
    'church', 'temple', 'museum',
}

# REFLECTION - passive processing
REFLECTION_LOCATIONS = {
    'hotel', 'hotel_room', 'hotel_lobby', 'hotel_ballroom',
    'house', 'apartment', 'bedroom', 'dorm',
}

# CORRECTION - intrusive intervention
CORRECTION_LOCATIONS = {
    'hospital', 'hospital_room', 'hospital_waiting',
    'clinic', 'doctor_office',
}

# TESTING/TEMPTATION - pleasure places that test appropriations
TESTING_LOCATIONS = {
    'casino',
    'amusement_park', 'waterpark', 'arcade', 'mall_arcade',
    'restaurant_buffet', 'restaurant_fast_food',
    'nightclub', 'bar',
}

# TRANSIT HUBS - places of transition/movement
TRANSIT_LOCATIONS = {
    'airport', 'airport_terminal', 'airport_gate', 'airport_security',
    'train_station', 'bus_station', 'subway',
    'parking_lot', 'parking_garage',
}

# REST - places of peace
REST_LOCATIONS = {
    'beach', 'forest', 'mountain', 'garden', 'park',
    'island', 'lake', 'river',
}

def categorize_function(loc_type):
    """Categorize location by spiritual function."""
    if loc_type in APPROPRIATION_LOCATIONS:
        return 'appropriation'
    elif loc_type in INSTRUCTION_LOCATIONS:
        return 'instruction'
    elif loc_type in REFLECTION_LOCATIONS:
        return 'reflection'
    elif loc_type in CORRECTION_LOCATIONS:
        return 'correction'
    elif loc_type in TESTING_LOCATIONS:
        return 'testing'
    elif loc_type in TRANSIT_LOCATIONS:
        return 'transit_hub'
    elif loc_type in REST_LOCATIONS:
        return 'rest'
    else:
        return 'other'

df_locations['functional_category'] = df_locations['location_type'].apply(categorize_function)

# Show distribution
print("FUNCTIONAL CATEGORY DISTRIBUTION")
print("="*60)
func_dist = df_locations['functional_category'].value_counts()
for cat, count in func_dist.items():
    print(f"  {cat:20s}: {count:,} ({count/len(df_locations)*100:.1f}%)")

FUNCTIONAL CATEGORY DISTRIBUTION
  other               : 4,436 (50.9%)
  appropriation       : 1,639 (18.8%)
  reflection          : 922 (10.6%)
  instruction         : 445 (5.1%)
  testing             : 414 (4.8%)
  transit_hub         : 409 (4.7%)
  rest                : 357 (4.1%)
  correction          : 85 (1.0%)


---

# HYPOTHESIS 1: Passive Transit → Instructional Destinations

**Prediction:** When transit is PASSIVE (being guided), the destination should be instructional (schools, places of teaching) or restful (places of peace).

In [4]:
print("="*70)
print("HYPOTHESIS 1: PASSIVE TRANSIT → INSTRUCTIONAL DESTINATIONS")
print("="*70)

# We need to link connections to their destination locations
# Each connection has to_location_id which maps to a location

# First, create a lookup for location functional categories by location_id
location_func_lookup = df_locations.set_index('location_id')['functional_category'].to_dict()
location_type_lookup = df_locations.set_index('location_id')['location_type'].to_dict()

# Add destination category to connections
df_connections['dest_function'] = df_connections['to_location_id'].map(location_func_lookup)
df_connections['dest_type'] = df_connections['to_location_id'].map(location_type_lookup)

# Filter to connections with known transit mode and destination
transit_analysis = df_connections[
    (df_connections['transit_mode'].isin(['passive', 'directed_active', 'wandering', 'drifting'])) &
    (df_connections['dest_function'].notna())
].copy()

print(f"\nConnections with transit mode AND destination function: {len(transit_analysis):,}")

# Cross-tabulate transit mode vs destination function
print("\n1.1 TRANSIT MODE → DESTINATION FUNCTION")
print("-" * 60)

contingency = pd.crosstab(transit_analysis['transit_mode'], transit_analysis['dest_function'])
print(contingency)

# Calculate percentages for passive transit specifically
print("\n1.2 WHERE DOES PASSIVE TRANSIT LEAD?")
print("-" * 60)
passive_conn = transit_analysis[transit_analysis['transit_mode'] == 'passive']
passive_dest = passive_conn['dest_function'].value_counts()
print(f"\nPassive transit destinations (n={len(passive_conn)}):")
for dest, count in passive_dest.items():
    print(f"  {dest:20s}: {count:,} ({count/len(passive_conn)*100:.1f}%)")

# Prediction check: instruction + rest should be higher for passive vs active
print("\n1.3 PREDICTION CHECK: Passive → Instruction/Rest")
print("-" * 60)

guidance_destinations = ['instruction', 'rest']

for mode in ['passive', 'directed_active', 'wandering']:
    mode_conn = transit_analysis[transit_analysis['transit_mode'] == mode]
    guidance_count = mode_conn[mode_conn['dest_function'].isin(guidance_destinations)].shape[0]
    total = len(mode_conn)
    if total > 0:
        rate = guidance_count / total * 100
        print(f"  {mode:20s}: {guidance_count:,}/{total:,} to instruction/rest ({rate:.1f}%)")

HYPOTHESIS 1: PASSIVE TRANSIT → INSTRUCTIONAL DESTINATIONS

Connections with transit mode AND destination function: 3,481

1.1 TRANSIT MODE → DESTINATION FUNCTION
------------------------------------------------------------
dest_function    appropriation  correction  instruction  other  reflection  \
transit_mode                                                                 
directed_active           1568           3           19    496         130   
drifting                    45           0            0     14           2   
passive                    308           1            0    103          27   
wandering                  220           1            2     69          16   

dest_function    rest  testing  transit_hub  
transit_mode                                 
directed_active    86       17          213  
drifting            3        0            5  
passive            17        8           54  
wandering           9        3           42  

1.2 WHERE DOES PASSIVE TRANSIT

In [5]:
# Statistical test: Is passive transit more likely to reach instruction/rest?
print("\n1.4 STATISTICAL TEST")
print("-" * 60)

# Binary: passive vs active, destination: guidance vs other
test_data = transit_analysis[
    transit_analysis['transit_mode'].isin(['passive', 'directed_active'])
].copy()

test_data['is_passive'] = test_data['transit_mode'] == 'passive'
test_data['is_guidance_dest'] = test_data['dest_function'].isin(['instruction', 'rest'])

contingency_test = pd.crosstab(test_data['is_passive'], test_data['is_guidance_dest'])
contingency_test.index = ['Active', 'Passive']
contingency_test.columns = ['Other Dest', 'Instruction/Rest']
print(contingency_test)

if contingency_test.shape == (2, 2) and contingency_test.sum().sum() > 20:
    chi2, p, dof, expected = chi2_contingency(contingency_test)
    print(f"\nχ² = {chi2:.2f}, df = {dof}, p = {p:.4f}")
    
    # Calculate rates
    passive_rate = contingency_test.loc['Passive', 'Instruction/Rest'] / contingency_test.loc['Passive'].sum() * 100
    active_rate = contingency_test.loc['Active', 'Instruction/Rest'] / contingency_test.loc['Active'].sum() * 100
    print(f"\nPassive → Instruction/Rest: {passive_rate:.1f}%")
    print(f"Active → Instruction/Rest: {active_rate:.1f}%")
    
    if passive_rate > active_rate:
        print("\n✓ SUPPORTS hypothesis: Passive transit leads MORE to instruction/rest")
    else:
        print("\n✗ Does not support: Rates are similar or reversed")


1.4 STATISTICAL TEST
------------------------------------------------------------
         Other Dest  Instruction/Rest
Active         2427               105
Passive         501                17

χ² = 0.63, df = 1, p = 0.4281

Passive → Instruction/Rest: 3.3%
Active → Instruction/Rest: 4.1%

✗ Does not support: Rates are similar or reversed


---

# HYPOTHESIS 2: Transit Success is Universal

**Prediction:** Both passive and active transit succeed at similar rates because they are *different mechanisms of the same functional system*, not competing approaches.

In [6]:
print("="*70)
print("HYPOTHESIS 2: TRANSIT SUCCESS IS UNIVERSAL")
print("="*70)

# Look at success rates by transit mode
outcome_analysis = df_connections[
    (df_connections['transit_mode'] != 'not_mentioned') &
    (df_connections['outcome'].isin(['succeeded', 'failed', 'partial']))
].copy()

print(f"\nConnections with transit mode AND clear outcome: {len(outcome_analysis):,}")

print("\n2.1 SUCCESS RATE BY TRANSIT MODE")
print("-" * 60)

for mode in ['passive', 'directed_active', 'wandering', 'drifting', 'fleeing', 'instant', 'struggle']:
    mode_conn = outcome_analysis[outcome_analysis['transit_mode'] == mode]
    if len(mode_conn) >= 10:
        success = (mode_conn['outcome'] == 'succeeded').sum()
        total = len(mode_conn)
        rate = success / total * 100
        print(f"  {mode:20s}: {success:,}/{total:,} succeeded ({rate:.1f}%)")

# Key comparison: passive vs directed_active
print("\n2.2 PASSIVE vs ACTIVE COMPARISON")
print("-" * 60)

passive_success = outcome_analysis[outcome_analysis['transit_mode'] == 'passive']
active_success = outcome_analysis[outcome_analysis['transit_mode'] == 'directed_active']

passive_rate = (passive_success['outcome'] == 'succeeded').sum() / len(passive_success) * 100
active_rate = (active_success['outcome'] == 'succeeded').sum() / len(active_success) * 100

print(f"  Passive success rate:  {passive_rate:.1f}%")
print(f"  Active success rate:   {active_rate:.1f}%")
print(f"  Difference:            {abs(passive_rate - active_rate):.1f} percentage points")

# Statistical test for equivalence
# If rates are within ~5%, they're functionally equivalent
if abs(passive_rate - active_rate) < 5:
    print("\n✓ SUPPORTS hypothesis: Success rates are essentially equivalent")
    print("  Both transit modes are functional mechanisms of the same system.")
else:
    print(f"\n⚠️ Notable difference of {abs(passive_rate - active_rate):.1f}%")

HYPOTHESIS 2: TRANSIT SUCCESS IS UNIVERSAL

Connections with transit mode AND clear outcome: 3,057

2.1 SUCCESS RATE BY TRANSIT MODE
------------------------------------------------------------
  passive             : 354/381 succeeded (92.9%)
  directed_active     : 1,778/1,858 succeeded (95.7%)
  wandering           : 236/246 succeeded (95.9%)
  drifting            : 57/59 succeeded (96.6%)
  fleeing             : 286/299 succeeded (95.7%)
  instant             : 160/162 succeeded (98.8%)
  struggle            : 39/52 succeeded (75.0%)

2.2 PASSIVE vs ACTIVE COMPARISON
------------------------------------------------------------
  Passive success rate:  92.9%
  Active success rate:   95.7%
  Difference:            2.8 percentage points

✓ SUPPORTS hypothesis: Success rates are essentially equivalent
  Both transit modes are functional mechanisms of the same system.


---

# HYPOTHESIS 3: Commercial Locations = Appropriation Zones

**Prediction:** Malls, markets, stores function as places of appropriating goods and truths. Libraries are rational appropriation through reasoning.

In [7]:
print("="*70)
print("HYPOTHESIS 3: COMMERCIAL LOCATIONS = APPROPRIATION")
print("="*70)

# Look at what happens in appropriation locations
appropriation_locs = df_locations[df_locations['functional_category'] == 'appropriation'].copy()

print(f"\nTotal appropriation locations: {len(appropriation_locs):,}")

print("\n3.1 APPROPRIATION LOCATION BREAKDOWN")
print("-" * 60)
approp_types = appropriation_locs['location_type'].value_counts()
for loc_type, count in approp_types.items():
    print(f"  {loc_type:25s}: {count:,}")

# Look at interactions in appropriation locations
print("\n3.2 INTERACTION TYPES IN APPROPRIATION LOCATIONS")
print("-" * 60)

# Get location IDs for appropriation locations
approp_loc_ids = set(appropriation_locs['location_id'].dropna())

# Filter interactions to appropriation locations
approp_interactions = df_interactions[df_interactions['location_id'].isin(approp_loc_ids)]

print(f"  Interactions in appropriation locations: {len(approp_interactions):,}")

if len(approp_interactions) > 0:
    inter_types = approp_interactions['interaction_type'].value_counts()
    print(f"\n  Interaction types:")
    for itype, count in inter_types.head(10).items():
        print(f"    {itype:25s}: {count:,}")

# Look at affective response in appropriation locations
print("\n3.3 EMOTIONAL RESPONSE IN APPROPRIATION LOCATIONS")
print("-" * 60)
affect_dist = appropriation_locs['affective_response'].value_counts()
for affect, count in affect_dist.head(10).items():
    print(f"  {affect:20s}: {count:,} ({count/len(appropriation_locs)*100:.1f}%)")

HYPOTHESIS 3: COMMERCIAL LOCATIONS = APPROPRIATION

Total appropriation locations: 1,639

3.1 APPROPRIATION LOCATION BREAKDOWN
------------------------------------------------------------
  mall                     : 879
  mall_store               : 590
  mall_food_court          : 112
  library                  : 58

3.2 INTERACTION TYPES IN APPROPRIATION LOCATIONS
------------------------------------------------------------
  Interactions in appropriation locations: 5,766

  Interaction types:
    observation              : 1,254
    navigation               : 1,132
    task                     : 782
    social                   : 771
    escape                   : 707
    search                   : 448
    conflict                 : 322
    transaction              : 212
    other                    : 138

3.3 EMOTIONAL RESPONSE IN APPROPRIATION LOCATIONS
------------------------------------------------------------
  not_mentioned       : 931 (56.8%)
  anxiety             : 237 (14.

---

# HYPOTHESIS 4: Hospitals = Corrective Intervention

**Prediction:** Hospital visits correlate with negative states being addressed — they are places of *intrusive correction*.

In [8]:
print("="*70)
print("HYPOTHESIS 4: HOSPITALS = CORRECTIVE INTERVENTION")
print("="*70)

# Look at hospital locations
hospital_locs = df_locations[df_locations['functional_category'] == 'correction'].copy()

print(f"\nTotal correction (hospital) locations: {len(hospital_locs):,}")

print("\n4.1 CORRECTION LOCATION BREAKDOWN")
print("-" * 60)
hospital_types = hospital_locs['location_type'].value_counts()
for loc_type, count in hospital_types.items():
    print(f"  {loc_type:25s}: {count:,}")

# Look at atmosphere in hospitals vs other locations
print("\n4.2 ATMOSPHERE IN HOSPITALS vs OTHER LOCATIONS")
print("-" * 60)

hospital_atmos = hospital_locs[hospital_locs['atmosphere'] != 'not_mentioned']['atmosphere'].value_counts()
other_atmos = df_locations[(df_locations['functional_category'] != 'correction') & 
                           (df_locations['atmosphere'] != 'not_mentioned')]['atmosphere'].value_counts()

print("HOSPITALS:")
for atmos, count in hospital_atmos.head(8).items():
    print(f"  {atmos:20s}: {count:,}")

print("\nOTHER LOCATIONS:")
for atmos, count in other_atmos.head(8).items():
    pct = count / other_atmos.sum() * 100
    print(f"  {atmos:20s}: {count:,} ({pct:.1f}%)")

# Look at interactions in hospitals
print("\n4.3 INTERACTIONS IN HOSPITALS")
print("-" * 60)

hospital_loc_ids = set(hospital_locs['location_id'].dropna())
hospital_interactions = df_interactions[df_interactions['location_id'].isin(hospital_loc_ids)]

print(f"  Total interactions in hospitals: {len(hospital_interactions):,}")

if len(hospital_interactions) > 0:
    inter_types = hospital_interactions['interaction_type'].value_counts()
    print(f"\n  Interaction types:")
    for itype, count in inter_types.head(10).items():
        print(f"    {itype:25s}: {count:,}")
    
    # Look at outcomes
    outcome_dist = hospital_interactions['outcome'].value_counts()
    print(f"\n  Interaction outcomes:")
    for outcome, count in outcome_dist.items():
        print(f"    {outcome:25s}: {count:,}")

HYPOTHESIS 4: HOSPITALS = CORRECTIVE INTERVENTION

Total correction (hospital) locations: 85

4.1 CORRECTION LOCATION BREAKDOWN
------------------------------------------------------------
  hospital                 : 60
  hospital_waiting         : 16
  hospital_room            : 9

4.2 ATMOSPHERE IN HOSPITALS vs OTHER LOCATIONS
------------------------------------------------------------
HOSPITALS:
  uncomfortable       : 13
  threatening         : 11
  oppressive          : 8
  eerie               : 8
  neutral             : 8
  nostalgic           : 1
  chaotic             : 1
  welcoming           : 1

OTHER LOCATIONS:
  neutral             : 857 (20.1%)
  threatening         : 702 (16.5%)
  eerie               : 644 (15.1%)
  uncomfortable       : 511 (12.0%)
  chaotic             : 495 (11.6%)
  welcoming           : 389 (9.1%)
  oppressive          : 226 (5.3%)
  peaceful            : 149 (3.5%)

4.3 INTERACTIONS IN HOSPITALS
----------------------------------------------------

---

# HYPOTHESIS 5: Pleasure Places = Testing/Temptation (The Illusion Pattern)

**Prediction:** Casinos, amusement parks, buffets appear outwardly good but reveal emptiness:
- Food may be tasteless
- Guidance may lead to negative outcomes
- Atmosphere shifts from welcoming to threatening
- The "all you can eat" abundance masks spiritual emptiness

In [9]:
print("="*70)
print("HYPOTHESIS 5: PLEASURE PLACES = TESTING/TEMPTATION")
print("="*70)

# Testing locations
testing_locs = df_locations[df_locations['functional_category'] == 'testing'].copy()

print(f"\nTotal testing/temptation locations: {len(testing_locs):,}")

print("\n5.1 TESTING LOCATION BREAKDOWN")
print("-" * 60)
testing_types = testing_locs['location_type'].value_counts()
for loc_type, count in testing_types.items():
    print(f"  {loc_type:25s}: {count:,}")

# The Illusion Pattern: Look for markers of deception/emptiness
print("\n5.2 THE ILLUSION PATTERN: Atmosphere")
print("-" * 60)

# Compare atmosphere in testing vs rest locations
rest_locs = df_locations[df_locations['functional_category'] == 'rest']

testing_atmos = testing_locs[testing_locs['atmosphere'] != 'not_mentioned']['atmosphere'].value_counts(normalize=True) * 100
rest_atmos = rest_locs[rest_locs['atmosphere'] != 'not_mentioned']['atmosphere'].value_counts(normalize=True) * 100

print("TESTING/TEMPTATION LOCATIONS:")
for atmos, pct in testing_atmos.head(8).items():
    print(f"  {atmos:20s}: {pct:.1f}%")

print("\nREST LOCATIONS (for comparison):")
for atmos, pct in rest_atmos.head(8).items():
    print(f"  {atmos:20s}: {pct:.1f}%")

# Key comparison: negative atmosphere rates
negative_atmospheres = ['threatening', 'oppressive', 'eerie', 'wrong', 'uncomfortable']
positive_atmospheres = ['welcoming', 'peaceful', 'nostalgic']

testing_negative = testing_locs[testing_locs['atmosphere'].isin(negative_atmospheres)].shape[0]
testing_positive = testing_locs[testing_locs['atmosphere'].isin(positive_atmospheres)].shape[0]
testing_with_atmos = testing_locs[testing_locs['atmosphere'] != 'not_mentioned'].shape[0]

rest_negative = rest_locs[rest_locs['atmosphere'].isin(negative_atmospheres)].shape[0]
rest_positive = rest_locs[rest_locs['atmosphere'].isin(positive_atmospheres)].shape[0]
rest_with_atmos = rest_locs[rest_locs['atmosphere'] != 'not_mentioned'].shape[0]

print(f"\n5.3 NEGATIVE vs POSITIVE ATMOSPHERE RATES")
print("-" * 60)
if testing_with_atmos > 0:
    print(f"TESTING: {testing_negative}/{testing_with_atmos} negative ({testing_negative/testing_with_atmos*100:.1f}%), {testing_positive}/{testing_with_atmos} positive ({testing_positive/testing_with_atmos*100:.1f}%)")
if rest_with_atmos > 0:
    print(f"REST:    {rest_negative}/{rest_with_atmos} negative ({rest_negative/rest_with_atmos*100:.1f}%), {rest_positive}/{rest_with_atmos} positive ({rest_positive/rest_with_atmos*100:.1f}%)")

HYPOTHESIS 5: PLEASURE PLACES = TESTING/TEMPTATION

Total testing/temptation locations: 414

5.1 TESTING LOCATION BREAKDOWN
------------------------------------------------------------
  amusement_park           : 116
  waterpark                : 114
  mall_arcade              : 62
  casino                   : 56
  restaurant_fast_food     : 54
  restaurant_buffet        : 12

5.2 THE ILLUSION PATTERN: Atmosphere
------------------------------------------------------------
TESTING/TEMPTATION LOCATIONS:
  chaotic             : 31.8%
  welcoming           : 14.1%
  neutral             : 14.1%
  eerie               : 11.8%
  uncomfortable       : 9.4%
  threatening         : 7.6%
  nostalgic           : 4.1%
  oppressive          : 2.9%

REST LOCATIONS (for comparison):
  threatening         : 25.8%
  peaceful            : 20.1%
  neutral             : 15.1%
  eerie               : 11.9%
  welcoming           : 7.5%
  chaotic             : 6.9%
  nostalgic           : 5.7%
  uncomfortable

In [10]:
# 5.4 Look at interactions in testing locations - do they show the illusion pattern?
print("\n5.4 INTERACTIONS IN TESTING LOCATIONS")
print("-" * 60)

testing_loc_ids = set(testing_locs['location_id'].dropna())
testing_interactions = df_interactions[df_interactions['location_id'].isin(testing_loc_ids)]

print(f"  Total interactions in testing locations: {len(testing_interactions):,}")

if len(testing_interactions) > 0:
    # Interaction types
    inter_types = testing_interactions['interaction_type'].value_counts()
    print(f"\n  Interaction types:")
    for itype, count in inter_types.head(10).items():
        print(f"    {itype:25s}: {count:,}")
    
    # Failure types (the illusion revealed)
    failure_dist = testing_interactions[testing_interactions['failure_type'] != 'not_applicable']['failure_type'].value_counts()
    if len(failure_dist) > 0:
        print(f"\n  Failure types (illusion revealed):")
        for ftype, count in failure_dist.items():
            print(f"    {ftype:25s}: {count:,}")
    
    # Outcomes
    outcome_dist = testing_interactions['outcome'].value_counts()
    print(f"\n  Interaction outcomes:")
    for outcome, count in outcome_dist.items():
        print(f"    {outcome:25s}: {count:,}")


5.4 INTERACTIONS IN TESTING LOCATIONS
------------------------------------------------------------
  Total interactions in testing locations: 5,704

  Interaction types:
    observation              : 1,248
    navigation               : 1,120
    task                     : 772
    social                   : 761
    escape                   : 702
    search                   : 431
    conflict                 : 321
    transaction              : 211
    other                    : 138

  Failure types (illusion revealed):
    environmental_block      : 224
    unclear                  : 156
    external_intervention    : 107
    no_effect                : 88
    physical_inability       : 53
    skill_failure            : 9

  Interaction outcomes:
    not_mentioned            : 1,820
    succeeded                : 1,781
    ongoing                  : 1,117
    failed                   : 587
    interrupted              : 333
    abandoned                : 66


In [11]:
# 5.5 Statistical test: Testing locations vs Rest locations - negative outcome rate
print("\n5.5 STATISTICAL TEST: Testing vs Rest")
print("-" * 60)

# Compare negative atmosphere rates
testing_has_neg = testing_locs['atmosphere'].isin(negative_atmospheres)
testing_has_pos = testing_locs['atmosphere'].isin(positive_atmospheres)
testing_valid = testing_locs['atmosphere'] != 'not_mentioned'

rest_has_neg = rest_locs['atmosphere'].isin(negative_atmospheres)
rest_has_pos = rest_locs['atmosphere'].isin(positive_atmospheres)
rest_valid = rest_locs['atmosphere'] != 'not_mentioned'

# Create contingency table: Location Type (Testing vs Rest) x Atmosphere (Negative vs Positive)
testing_data = pd.DataFrame({
    'location_type': ['Testing'] * testing_valid.sum() + ['Rest'] * rest_valid.sum(),
    'atmosphere_valence': (
        ['negative'] * testing_has_neg.sum() + 
        ['positive'] * testing_has_pos.sum() + 
        ['neutral'] * (testing_valid.sum() - testing_has_neg.sum() - testing_has_pos.sum()) +
        ['negative'] * rest_has_neg.sum() + 
        ['positive'] * rest_has_pos.sum() + 
        ['neutral'] * (rest_valid.sum() - rest_has_neg.sum() - rest_has_pos.sum())
    )
})

# Simplified: just compare negative rates
testing_neg_rate = testing_has_neg.sum() / testing_valid.sum() * 100 if testing_valid.sum() > 0 else 0
rest_neg_rate = rest_has_neg.sum() / rest_valid.sum() * 100 if rest_valid.sum() > 0 else 0

print(f"Negative atmosphere rate in TESTING locations: {testing_neg_rate:.1f}%")
print(f"Negative atmosphere rate in REST locations:    {rest_neg_rate:.1f}%")

# Fisher's exact test
if testing_valid.sum() > 10 and rest_valid.sum() > 10:
    contingency = pd.DataFrame({
        'Negative': [testing_has_neg.sum(), rest_has_neg.sum()],
        'Non-Negative': [testing_valid.sum() - testing_has_neg.sum(), rest_valid.sum() - rest_has_neg.sum()]
    }, index=['Testing', 'Rest'])
    print(f"\nContingency table:")
    print(contingency)
    
    odds, p = fisher_exact(contingency)
    print(f"\nFisher's exact test:")
    print(f"  Odds ratio = {odds:.2f}")
    print(f"  p = {p:.4f}")
    
    if p < 0.05 and odds > 1:
        print("\n✓ SUPPORTS hypothesis: Testing locations have MORE negative atmosphere")
        print("  This supports the 'illusion pattern' - places that appear good but reveal emptiness.")
    elif p < 0.05 and odds < 1:
        print("\n✗ Opposite: Testing locations have LESS negative atmosphere")
    else:
        print("\n= No significant difference")


5.5 STATISTICAL TEST: Testing vs Rest
------------------------------------------------------------
Negative atmosphere rate in TESTING locations: 33.5%
Negative atmosphere rate in REST locations:    44.7%

Contingency table:
         Negative  Non-Negative
Testing        57           113
Rest           71            88

Fisher's exact test:
  Odds ratio = 0.63
  p = 0.0422

✗ Opposite: Testing locations have LESS negative atmosphere


---

# HYPOTHESIS 6: Baseline Reality is Solid

**Prediction:** Most locations show stable reality because the structure is inherent to the World of Spirits, not projected by the experiencer.

In [12]:
print("="*70)
print("HYPOTHESIS 6: BASELINE REALITY IS SOLID")
print("="*70)

# Overall reality stability distribution
reality_dist = df_locations[df_locations['reality_stability'] != 'not_mentioned']['reality_stability'].value_counts()

print(f"\n6.1 OVERALL REALITY STABILITY DISTRIBUTION")
print("-" * 60)
total_with_stability = reality_dist.sum()
for stability, count in reality_dist.items():
    print(f"  {stability:20s}: {count:,} ({count/total_with_stability*100:.1f}%)")

# Calculate solid rate
solid_count = reality_dist.get('solid', 0)
hyper_real_count = reality_dist.get('hyper_real', 0)
stable_count = solid_count + hyper_real_count

unstable_types = ['plastic', 'shifting', 'decaying']
unstable_count = sum(reality_dist.get(ut, 0) for ut in unstable_types)

print(f"\n6.2 STABLE vs UNSTABLE")
print("-" * 60)
print(f"  STABLE (solid + hyper_real): {stable_count:,} ({stable_count/total_with_stability*100:.1f}%)")
print(f"  UNSTABLE (plastic + shifting + decaying): {unstable_count:,} ({unstable_count/total_with_stability*100:.1f}%)")

# Binomial test: is stable rate > 50%?
if stable_count + unstable_count > 20:
    result = binomtest(stable_count, stable_count + unstable_count, 0.5, alternative='greater')
    print(f"\n  Binomial test (H0: stable rate = 50%): p = {result.pvalue:.6f}")
    if result.pvalue < 0.001:
        print("  ⚠️ HIGHLY SIGNIFICANT: Reality is predominantly STABLE")
        print("\n✓ STRONGLY SUPPORTS hypothesis: Baseline reality is solid.")
        print("  The World of Spirits has inherent structure, not projected instability.")

HYPOTHESIS 6: BASELINE REALITY IS SOLID

6.1 OVERALL REALITY STABILITY DISTRIBUTION
------------------------------------------------------------
  solid               : 2,948 (77.5%)
  shifting            : 623 (16.4%)
  hyper_real          : 180 (4.7%)
  plastic             : 44 (1.2%)
  decaying            : 9 (0.2%)

6.2 STABLE vs UNSTABLE
------------------------------------------------------------
  STABLE (solid + hyper_real): 3,128 (82.2%)
  UNSTABLE (plastic + shifting + decaying): 676 (17.8%)

  Binomial test (H0: stable rate = 50%): p = 0.000000
  ⚠️ HIGHLY SIGNIFICANT: Reality is predominantly STABLE

✓ STRONGLY SUPPORTS hypothesis: Baseline reality is solid.
  The World of Spirits has inherent structure, not projected instability.


In [13]:
# 6.3 Reality stability by functional category
print("\n6.3 REALITY STABILITY BY FUNCTIONAL CATEGORY")
print("-" * 60)

for category in ['instruction', 'appropriation', 'reflection', 'correction', 'testing', 'rest']:
    cat_locs = df_locations[
        (df_locations['functional_category'] == category) &
        (df_locations['reality_stability'] != 'not_mentioned')
    ]
    if len(cat_locs) >= 20:
        solid = (cat_locs['reality_stability'] == 'solid').sum()
        total = len(cat_locs)
        solid_rate = solid / total * 100
        print(f"  {category:15s}: {solid:,}/{total:,} solid ({solid_rate:.1f}%)")


6.3 REALITY STABILITY BY FUNCTIONAL CATEGORY
------------------------------------------------------------
  instruction    : 127/170 solid (74.7%)
  appropriation  : 565/705 solid (80.1%)
  reflection     : 315/444 solid (70.9%)
  correction     : 37/48 solid (77.1%)
  testing        : 116/153 solid (75.8%)
  rest           : 145/172 solid (84.3%)


---

# SUMMARY: Corrected Hypothesis Results

In [14]:
print("="*70)
print("CORRECTED HYPOTHESIS TESTING: SUMMARY")
print("="*70)

print("""
HYPOTHESIS 1: Passive Transit → Instructional Destinations
─────────────────────────────────────────────────────────────
Prediction: Passive (guided) transit leads to teaching/rest
Status: [See detailed results above]

HYPOTHESIS 2: Transit Success is Universal
─────────────────────────────────────────────────────────────
Prediction: Both passive and active transit succeed equally
because they are different mechanisms of the same system.
Status: [See detailed results above]

HYPOTHESIS 3: Commercial Locations = Appropriation
─────────────────────────────────────────────────────────────
Prediction: Malls/markets function for acquiring goods/truths
Status: [See detailed results above]

HYPOTHESIS 4: Hospitals = Corrective Intervention
─────────────────────────────────────────────────────────────
Prediction: Hospitals are places of intrusive correction
Status: [See detailed results above]

HYPOTHESIS 5: Pleasure Places = Testing/Temptation
─────────────────────────────────────────────────────────────
Prediction: Casinos/amusement parks show the illusion pattern
(appear good, reveal emptiness)
Status: [See detailed results above]

HYPOTHESIS 6: Baseline Reality is Solid
─────────────────────────────────────────────────────────────
Prediction: Most locations are stable because structure is
inherent to the World of Spirits, not projected.
Status: [See detailed results above]
""")

print("\n" + "="*70)
print("KEY INSIGHT")
print("="*70)
print("""
The original hypotheses treated the spiritual world as REACTIVE to the
experiencer's emotional state. The corrected framework recognizes it as
having INHERENT FUNCTIONAL STRUCTURE:

• Transit modes are different mechanisms, not competing approaches
• Locations serve specific spiritual functions (appropriation,
  instruction, correction, testing, reflection, rest)
• Reality is fundamentally SOLID — the structure exists independently
• The "illusion pattern" reveals places that appear good but are empty
""")

CORRECTED HYPOTHESIS TESTING: SUMMARY

HYPOTHESIS 1: Passive Transit → Instructional Destinations
─────────────────────────────────────────────────────────────
Prediction: Passive (guided) transit leads to teaching/rest
Status: [See detailed results above]

HYPOTHESIS 2: Transit Success is Universal
─────────────────────────────────────────────────────────────
Prediction: Both passive and active transit succeed equally
because they are different mechanisms of the same system.
Status: [See detailed results above]

HYPOTHESIS 3: Commercial Locations = Appropriation
─────────────────────────────────────────────────────────────
Prediction: Malls/markets function for acquiring goods/truths
Status: [See detailed results above]

HYPOTHESIS 4: Hospitals = Corrective Intervention
─────────────────────────────────────────────────────────────
Prediction: Hospitals are places of intrusive correction
Status: [See detailed results above]

HYPOTHESIS 5: Pleasure Places = Testing/Temptation
──────────